# Benchmarking Ordering Policies\n\nThis notebook compares different ordering policies using the DeepBullwhip benchmarking framework.\n\nWe compare:\n- **Order-Up-To (OUT)**: Standard base-stock policy\n- **POUT (alpha=0.3)**: Proportional OUT with aggressive smoothing\n- **POUT (alpha=0.7)**: Proportional OUT with moderate smoothing\n- **Constant Order**: Fixed quantity baseline (BWR=0 by construction)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from deepbullwhip.benchmark import BenchmarkRunner

runner = BenchmarkRunner(
    chain_config="semiconductor_4tier",
    demand="semiconductor_ar1",
    T=156,
    N=50,  # 50 Monte Carlo paths for quick demo
    seed=42,
)

results = runner.run(
    policies=[
        "order_up_to",
        ("proportional_out", {"alpha": 0.3}),
        ("proportional_out", {"alpha": 0.7}),
        ("constant_order", {"order_quantity": 11.6}),
    ],
    forecasters=["naive"],
    metrics=["BWR", "FILL_RATE", "TC"],
)

print(results.to_string(index=False))

## Pivot Table: Policy x Metric

In [ ]:
pivot = results.pivot_table(
    index=["policy", "echelon"],
    columns="metric",
    values="value",
    aggfunc="mean",
)
print(pivot.to_string(float_format="%.3f"))

## Comparison vs OUT Baseline

In [ ]:
comparison = runner.compare(results, baseline="order_up_to")
comp_pivot = comparison.pivot_table(
    index=["policy", "echelon"],
    columns="metric",
    values="pct_change",
    aggfunc="mean",
)
print("Percentage change vs OUT baseline:")
print(comp_pivot.to_string(float_format="%+.1f%%"))

## BWR vs Fill Rate Tradeoff (Echelon 1)\n\nThe key tradeoff: lower alpha reduces bullwhip but may reduce fill rate.

In [ ]:
e1 = results[results["echelon"] == "E1"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for metric, ax in zip(["BWR", "FILL_RATE"], axes):
    data = e1[e1["metric"] == metric]
    ax.barh(data["policy"], data["value"])
    ax.set_xlabel(metric)
    ax.set_title(f"{metric} at Echelon 1")

plt.tight_layout()
plt.savefig("benchmark_policies_tradeoff.png", dpi=150, bbox_inches="tight")
plt.show()

## Export LaTeX Table

In [ ]:
from deepbullwhip.benchmark.report import to_latex

latex = to_latex(results, caption="Policy Comparison on Semiconductor 4-Tier Chain", label="tab:policy-comparison")
print(latex)